# BERT Masking and Masked-LM Loss
Pure NumPy implementation of BERT-style 80/10/10 corruption and masked-only cross-entropy loss.

In [1]:
import numpy as np

tokens = list(range(10, 22))
positions = list(range(10))

In [2]:
def apply_masking(tokens, positions, mask_id, vocab_size, seed=0):
    rng = np.random.RandomState(seed)
    masked_tokens = list(tokens)
    labels = [-100] * len(tokens)

    for p in sorted(positions):
        original_token = tokens[p]
        labels[p] = original_token

        r = rng.rand()
        if r < 0.8:
            masked_tokens[p] = mask_id
        elif r < 0.9:
            masked_tokens[p] = int(rng.randint(0, vocab_size))
        else:
            masked_tokens[p] = original_token

    return masked_tokens, labels

In [3]:
def mlm_loss(logits, labels, ignore_index=-100):
    logits = np.asarray(logits, dtype=float)
    labels = np.asarray(labels)
    valid = labels != ignore_index

    if not np.any(valid):
        return 0.0

    selected_logits = logits[valid]
    selected_labels = labels[valid].astype(int)

    max_logits = np.max(selected_logits, axis=-1, keepdims=True)
    shifted = selected_logits - max_logits
    log_probs = shifted - np.log(np.sum(np.exp(shifted), axis=-1, keepdims=True))
    losses = -log_probs[np.arange(len(selected_labels)), selected_labels]
    return float(np.mean(losses))

In [4]:
masked, labels = apply_masking(
    list(range(10, 22)), list(range(10)), 103, 1000, seed=0
)
print(sum(label != -100 for label in labels))

V = 50
print(round(mlm_loss(np.zeros((3, V)), np.array([-100, 7, -100])), 4))
print(mlm_loss(np.zeros((3, V)), np.full(3, -100)))

10
3.912
0.0
